# 01 — Cached episodes and frozen DINOv2 baselines

This notebook is a thin Colab entry point. Reusable code lives in `src/cross_image_glot`.

In [36]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph")

if "<YOUR_GITHUB_USERNAME>" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository before running this notebook.")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImageGLOT_repo
!pip install -q -r requirements.txt

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source directory:", SRC_DIR)
print("Source exists:", SRC_DIR.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 417 bytes | 417.00 KiB/s, done.
From https://github.com/TomerBurman/CrossImagePatchGraph
   178ccd9..3827ffb  main       -> origin/main
Updating 178ccd9..3827ffb
error: Your local changes to the following files would be overwritten by merge:
	src/cross_image_glot/config.py
Please commit your changes or stash them before you merge.
Aborting
/content/CrossImageGLOT_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cross-image-glot (pyproject.toml) ... 

In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)

In [34]:
from cross_image_glot.storage import restore_feature_splits
from cross_image_glot.data import MiniImageNetFeatureDataset, FewShotFeatureEpisodeDataset, validate_episode_disjointness
from cross_image_glot.baselines import evaluate_frozen_baseline
from cross_image_glot.storage import atomic_json_save

restore_feature_splits(["val"], paths.drive_feature_dir, paths.local_feature_dir)
val_features = MiniImageNetFeatureDataset(paths.local_feature_dir, "val", max_cached_shards=6)
val_episodes = FewShotFeatureEpisodeDataset(
    val_features, n_way=5, k_shot=5, queries_per_class=15,
    num_episodes=600, seed=10_000, vary_by_epoch=False,
)
episode = val_episodes[0]
validate_episode_disjointness(episode)
print("Support patches:", episode["support_patches"].shape)
print("Query patches:", episode["query_patches"].shape)

FileNotFoundError: Persistent feature metadata is missing: /content/drive/MyDrive/CrossImageGLOT/features/dinov2_vits14_224/metadata.json

In [ ]:
cls_metrics = evaluate_frozen_baseline(val_episodes, "cls", device, num_episodes=100, temperature=0.1)
mean_patch_metrics = evaluate_frozen_baseline(val_episodes, "mean_patch", device, num_episodes=100, temperature=0.1)
print("CLS:", cls_metrics)
print("Mean patch:", mean_patch_metrics)

result = {"cls": cls_metrics.to_dict(), "mean_patch": mean_patch_metrics.to_dict()}
output = paths.drive_results_dir / "frozen_baselines_5way5shot.json"
atomic_json_save(result, output)
print("Saved:", output)